<a href="https://colab.research.google.com/github/harrisonritz/CCN2026_SSM-tutorial/blob/main/02_lds_parameter_recovery_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CCN Tutorial · Notebook 2 — R / MARSS translation
## Simulate → fit → **recover**: parameter estimation and the (non-)identifiability of linear dynamical systems

*R port of the Python/`dynamax` notebook, built on the [MARSS](https://atsa-es.github.io/MARSS/) package (Multivariate Autoregressive State-Space modeling).*

In Notebook 1 the parameters $\theta=(A,C,Q,R,m_0,V_0)$ were **known**, and we did inference over states. Here we do the opposite: we observe only $y_{1:T}$ and must **learn $\theta$**. The workhorse is Expectation–Maximization (EM), and the two halves map onto objects from Notebook 1:

- **E-step** = run a Kalman **smoother** with the current $\theta$ to get the posterior over latents (using *all* of $y_{1:T}$);
- **M-step** = update $\theta$ in closed form from those posterior expectations.

We call MARSS's EM implementation — `MARSS(..., method = "kem")`. The scientific point of this notebook is what comes *after* fitting: **a state-space model is only identifiable up to a change of latent basis**, so "did we recover the truth?" is subtler than it looks. We'll see what is recoverable (eigenvalues, the Hankel spectrum, the latent trajectory *after alignment*, the model order, and — once each model is put into a canonical basis — every parameter matrix) and what is not (the parameters in the basis EM happens to return them in, the latent coordinate system itself).

> **Notation — mind the letters.** We use the dynamical-systems convention: $A$ = dynamics matrix, $C$ = emission/loading matrix. **MARSS uses different symbols:** it writes the dynamics matrix as $B$ and the emission/loading matrix as $Z$. So everywhere below, our `A` is handed to MARSS as `B`, and our `C` is handed to MARSS as `Z`. (Confusingly, MARSS's own `A` is the *observation intercept* — a different object — which we fix to zero.)

> **Three structural differences from the Python notebook.**
> 1. **One series, not a batch.** `dynamax` fits a *batch* of independent sequences at once. MARSS instead fits a **single** multivariate time series (rows = channels, columns = time), so we simulate one long training series and one long validation series rather than 25×250 independent trials.
> 2. **A smaller readout.** The Python notebook uses $d_y=50$ channels. MARSS's EM is far slower than `dynamax`'s, and Section 9 refits the model ten times, so we use $d_y=20$. Every structural feature of the generative model — non-normal dynamics, heteroskedastic process noise, spatially correlated observation noise, fixed SNR — is carried over unchanged.
> 3. **The estimated noise structure.** `dynamax`'s conjugate LG-SSM estimates *full* $Q$ and $R$. Here we ask MARSS for `Q, R = "diagonal and unequal"`. That matches the generating $Q$ exactly, but it is deliberately **misspecified** for $R$, whose truth is spatially correlated — which is why Section 8 reports a visibly nonzero error for $R$ and a small one for the rest. Fitting an unconstrained $20\times20$ $R$ (210 free parameters) with EM is not practical in a tutorial.
>
> The identifiability story — and every invariant we compute — is unchanged by any of this.

## 0. Setup

**Running on Colab:** choose an **R** runtime (*Runtime → Change runtime type → R*); this notebook's kernel is already set to R. The first `install.packages("MARSS")` can take a minute or two.

In [ ]:
## Install dependencies if missing (safe to re-run)
if (!requireNamespace("MARSS", quietly = TRUE)) {
    install.packages("MARSS")
}
if (!requireNamespace("MASS", quietly = TRUE)) {
    install.packages("MASS")
}

library(MARSS) # state-space fitting (EM / BFGS), Kalman filter & smoother
library(MASS) # mvrnorm for the ground-truth simulator

set.seed(1)
options(digits = 4)

## ---------------------------------------------------------------- helpers
## Base R has no scipy, so the three numerical primitives this notebook leans
## on are defined once here and reused throughout.

## sort complex numbers by (Re, then Im) so true & fitted spectra line up
sort_complex <- function(z) z[order(Re(z), Im(z))]

## discrete Lyapunov  X = A X A' + W  by vectorization,
##     vec(X) = (I - A (x) A)^-1 vec(W).
## Used for the stationary latent covariance (Section 2), the Hankel factors
## (Section 7) and both Gram matrices (Section 8).
dlyap <- function(A, W) {
    d <- nrow(A)
    matrix(solve(diag(d * d) - kronecker(A, A), as.vector(W)), d, d)
}

## integer matrix power
mpow <- function(M, k) {
    P <- diag(nrow(M))
    if (k > 0) {
        for (i in seq_len(k)) {
            P <- P %*% M
        }
    }
    P
}

## relative Frobenius error -- zero iff the two matrices agree exactly
rel_fro <- function(P, Phat) norm(Phat - P, "F") / norm(P, "F")

## 1. The model, and the symmetry that haunts it

The linear–Gaussian SSM, as in Notebook 1:
$$
x_t = A\,x_{t-1} + w_t,\quad w_t\sim\mathcal N(0,Q);\qquad
y_t = C\,x_t + v_t,\quad v_t\sim\mathcal N(0,R);\qquad x_1\sim\mathcal N(m_0,V_0).
$$

Here is the fact that shapes everything below. Pick **any** invertible matrix $T\in\mathrm{GL}(d_x)$ and relabel the latent state $x_t \mapsto T x_t$. Absorbing $T$ into the parameters,
$$
A \mapsto T A T^{-1},\quad
C \mapsto C T^{-1},\quad
Q \mapsto T Q T^{\top},\quad
m_0 \mapsto T m_0,\quad
V_0 \mapsto T V_0 T^{\top},\quad
R \mapsto R,
$$
gives a **different** parameter set inducing the **identical** distribution over the observations $p(y_{1:T})$. The likelihood is exactly flat along this $d_x^2$-dimensional group orbit, so no amount of data can pin down *which* member of the orbit generated the data.

**Consequences.**
- Comparing a fitted $\hat A$ to the true $A$ entry-by-entry is meaningless — they can differ wildly yet describe the same system.
- Comparing fitted latent trajectories to true ones requires first **undoing** the basis change (alignment).
- What *is* estimable are quantities **invariant** to $T$. Two we will use: the **eigenvalues of $A$** (a similarity transform preserves them) and the **Hankel matrix of output covariances** (a property of $y$ itself).

## 2. A ground-truth system

We build a $d_x=5$ latent with a deliberately readable spectrum — two **complex-conjugate pairs** (damped rotations of period $\approx20$ and $\approx8$ steps, with $|\lambda|=0.95$ and $0.85$) plus one leftover **real** mode ($\lambda=0.9$, slow decay) — then rotate the coordinates by a random orthogonal matrix so the modes are mixed across all latent dimensions (nothing is secretly axis-aligned). It is read out into $d_y=20$ noisy channels.

Three ingredients make this harder, and more realistic, than a toy:

- **Non-normal dynamics.** Feedforward coupling is placed strictly *above* the modal block-diagonal, so $A$ stays block-upper-triangular: the eigenvalues — and hence stability — are exactly unchanged, but the eigenvectors become oblique and the system **amplifies transiently** before decaying (balanced amplification). Set `nonnormal <- FALSE` for the normal case.
- **Heteroskedastic process noise.** $Q$ is diagonal but far from a multiple of the identity, so no latent direction is privileged by construction.
- **Structured observation noise.** Channel variances differ (one is deliberately terrible), *and* channels are spatially autocorrelated, $R = D^{1/2}KD^{1/2}$ with $K_{ij}=\exp(-|i-j|/\ell)$. Correlated noise concentrates into a few large eigenvalues — which is exactly how it can masquerade as extra latent dimensions. The printed **effective rank** of $R$ says how many channels' worth of noise you are really up against.

Finally, the loadings $C$ are rescaled to hit a fixed per-channel **signal-to-noise ratio**. Without that, latent power swings by orders of magnitude as $d_x$, the feedforward gain, or $\ell$ change (non-normal amplification alone moves it $\sim1000\times$), and nothing is comparable across settings.

In [ ]:
dx <- 5L
dy <- 20L

## ------------------------------------------------------------ latent dynamics
## Rotational modes occupy coordinate *pairs*: (1,2), (3,4), ... With dx odd the
## leftover coordinate is a single real mode. Non-normality is feedforward coupling
## placed strictly ABOVE the modal block-diagonal, which keeps A block-upper-
## triangular: the eigenvalues -- and hence stability -- are exactly unchanged,
## while the eigenvectors become oblique and the system amplifies transiently
## before decaying (functionally feedforward / balanced amplification).
## Set nonnormal = FALSE for the normal (orthogonal-mode) case.
nonnormal <- TRUE
ff_gain <- 1.0
ff_mode <- "chain" # "last": only the final block drives the others (gentle);
## "chain": each block drives the previous one, so amplification compounds with dx.
real_eig <- 0.9 # eigenvalue of the leftover real mode (odd dx only)
periods <- c(20, 8) # rotation period of the slowest / fastest pair
radii <- c(0.95, 0.85) # |lambda| of the slowest / fastest pair

n_pair <- dx %/% 2
per <- seq(periods[1], periods[2], length.out = n_pair)
rad <- seq(radii[1], radii[2], length.out = n_pair)
A_modal <- matrix(0, dx, dx)
blocks <- list() # (start, stop) of each diagonal block, in order
for (k in seq_len(n_pair)) {
    th <- 2 * pi / per[k]
    s <- 2 * (k - 1)
    A_modal[(s + 1):(s + 2), (s + 1):(s + 2)] <-
        rad[k] * matrix(c(cos(th), sin(th), -sin(th), cos(th)), 2, 2)
    blocks[[length(blocks) + 1L]] <- c(s + 1L, s + 2L)
}
if (dx %% 2 == 1) {
    A_modal[dx, dx] <- real_eig
    blocks[[length(blocks) + 1L]] <- c(dx, dx)
}

if (nonnormal && length(blocks) > 1) {
    # a single block cannot be made non-normal
    if (ff_mode == "chain") {
        for (k in seq_len(length(blocks) - 1L)) {
            a <- blocks[[k]]
            b <- blocks[[k + 1L]]
            A_modal[a[1]:a[2], b[1]:b[2]] <- ff_gain # block k+1 drives block k
        }
    } else {
        lo <- blocks[[length(blocks)]][1]
        A_modal[1:(lo - 1L), lo:dx] <- ff_gain # last block drives everything upstream
    }
}

M <- qr.Q(qr(matrix(rnorm(dx * dx), dx, dx))) # random orthogonal mixing
A <- M %*% A_modal %*% t(M) # dynamics matrix  (MARSS calls this B)

Q <- 0.1 * diag(dx) + 0.9 * diag(rnorm(dx)^2, nrow = dx) # heteroskedastic noise
C <- matrix(rnorm(dy * dx), dy, dx) # emission / loading matrix (MARSS: Z)
C <- sweep(C, 2, sqrt(colSums(C^2)), "/") # unit-norm columns
snr <- 0.5 # per-channel signal-to-noise ratio; NULL keeps C exactly as drawn

## --------------------------------------------------------- observation noise
## Channels differ in variance AND are spatially autocorrelated: R = D^(1/2) K D^(1/2)
## with K_ij = exp(-d_ij / obs_ell). That Toeplitz K is positive definite for any
## obs_ell > 0, and its inverse is tridiagonal -- i.e. this is exactly the Gaussian
## Markov random field whose precision is a chain-graph Laplacian, so "Toeplitz" and
## "graph Laplacian" are the same object here. obs_ell = 0 gives uncorrelated channels.
obs_ell <- 5.0 # spatial correlation length, in channels (0 = white)
obs_ring <- FALSE # TRUE: channels sit on a ring, so distance wraps around

chan_var <- 0.1 + 0.9 * rnorm(dy)^2 # heteroskedastic channel variances
chan_var[1] <- chan_var[1] + sqrt(sum(chan_var)) # one deliberately terrible channel
d_ij <- abs(outer(seq_len(dy), seq_len(dy), "-"))
if (obs_ring) {
    d_ij <- pmin(d_ij, dy - d_ij)
}
K_corr <- if (obs_ell > 0) exp(-d_ij / obs_ell) else diag(dy)
R <- K_corr * sqrt(outer(chan_var, chan_var))

## Fix the signal-to-noise ratio explicitly. Without this, the latent power swings by
## orders of magnitude as dx / ff_gain / obs_ell change (non-normal amplification alone
## moves it ~1000x), so nothing is comparable across settings.
Pi_true <- dlyap(A, Q)
if (!is.null(snr)) {
    C <- C * sqrt(snr * mean(diag(R)) / mean(diag(C %*% Pi_true %*% t(C))))
}

m_0 <- rnorm(dx)
V_0 <- 2 * diag(dx)

eig_true <- sort_complex(eigen(A)$values)
cat("true eigenvalues of A:\n")
print(eig_true)
cat("|lambda| =", round(Mod(eig_true), 3), "\n")

## Stability is set by the spectrum; transient amplification is set by non-normality.
gain <- sapply(1:80, function(t) norm(mpow(A, t), "2"))
cat(sprintf(
    "spectral radius = %.3f  (stable: %s)\n",
    max(Mod(eig_true)),
    max(Mod(eig_true)) < 1
))
cat(sprintf(
    "non-normality ||A'A - AA'||_F = %.2f\n",
    norm(t(A) %*% A - A %*% t(A), "F")
))
cat(sprintf(
    "peak transient gain max_t ||A^t||_2 = %.2f at t = %d\n",
    max(gain),
    which.max(gain)
))

## How separable are signal and noise? Correlated noise concentrates into a few large
## eigenvalues, which is exactly how it can masquerade as extra latent dimensions.
sig_ev <- eigen(
    C %*% Pi_true %*% t(C),
    symmetric = TRUE,
    only.values = TRUE
)$values[1:dx]
noise_ev <- eigen(R, symmetric = TRUE, only.values = TRUE)$values
eff_rank <- sum(noise_ev)^2 / sum(noise_ev^2)
cat("\nsignal eigenvalues (C Pi C') =", round(sig_ev, 2), "\n")
cat("top noise eigenvalues of R   =", round(noise_ev[1:3], 2), "\n")
cat(sprintf("effective rank of R = %.1f of %d channels\n", eff_rank, dy))

In [ ]:
## ------------------------------------------------------------------ Figure 1a
## The parameters themselves, as heat maps. One panel per matrix, each on its own
## color scale (magnitudes differ several-fold, so a shared scale would flatten Q
## to nothing). Signed matrices get a diverging map centered at zero; non-negative
## ones a sequential map anchored at zero.
##
## Canvas first: Jupyter/Colab hand base R a repr.plot.width x repr.plot.height inch
## device, and the default 7 x 7 is cramped for eight layout columns. Ask for a wider
## one here, then put the option back so the rest of the notebook is unaffected.
old_repr <- options(repr.plot.width = 14, repr.plot.height = 8)

div_cols <- hcl.colors(64, "Blue-Red 3")
seq_cols <- hcl.colors(64, "Viridis")

show_mat <- function(M, title, xlabs = NULL, ylabs = NULL, annot = FALSE) {
    M <- as.matrix(M)
    nr <- nrow(M)
    nc <- ncol(M)
    if (min(M) < 0) {
        v <- max(abs(M))
        zl <- c(-v, v)
        cols <- div_cols
    } else {
        zl <- c(0, max(max(M), 1e-12))
        cols <- seq_cols
    }
    ## image() wants z[x, y]; transpose and flip so row 1 is drawn at the top
    image(
        x = seq_len(nc),
        y = seq_len(nr),
        z = t(M)[, nr:1, drop = FALSE],
        zlim = zl,
        col = cols,
        axes = FALSE,
        xlab = "",
        ylab = "",
        main = title,
        cex.main = 0.95
    )
    box()
    tick <- function(n, lbls, side) {
        if (is.null(lbls)) {
            # plain 1-based indices, thinned when there are many
            at <- seq(1, n, by = if (n <= 6) 1 else ceiling(n / 6))
            lbls <- as.character(at)
        } else {
            at <- seq_len(n)
        }
        if (side == 1) {
            axis(
                1,
                at = at,
                labels = lbls,
                tick = FALSE,
                line = -0.7,
                cex.axis = 0.7
            )
        } else {
            axis(
                2,
                at = nr - at + 1,
                labels = lbls,
                tick = FALSE,
                las = 1,
                line = -0.7,
                cex.axis = 0.7
            )
        }
    }
    tick(nc, xlabs, 1)
    tick(nr, ylabs, 2)
    if (annot) {
        for (i in seq_len(nr)) {
            for (j in seq_len(nc)) {
                ## keep the text readable on any cell
                sh <- (M[i, j] - zl[1]) / diff(zl)
                rgbv <- col2rgb(cols[max(1, min(64, ceiling(sh * 64)))]) / 255
                lum <- 0.30 * rgbv[1] + 0.59 * rgbv[2] + 0.11 * rgbv[3]
                text(
                    j,
                    nr - i + 1,
                    sprintf("%.2f", M[i, j]),
                    col = if (lum > 0.55) "black" else "white",
                    cex = 0.65
                )
            }
        }
    }
    invisible(list(zl = zl, cols = cols))
}

## The colorbar sits in a deliberately narrow layout column, and base R refuses to
## open a figure whose margins do not fit inside its cell ("figure margins too
## large") -- an axis() colorbar therefore dies on any smallish device. So take no
## margins at all and draw the ramp, its frame and its tick labels as ordinary
## graphics inside a 0-1 box. Nothing here depends on the device size.
color_bar <- function(info) {
    op <- par(mar = c(0, 0, 0, 0))
    plot.new()
    plot.window(xlim = c(0, 1), ylim = c(0, 1), xaxs = "i", yaxs = "i")
    n <- length(info$cols)
    y <- seq(0.12, 0.88, length.out = n + 1) # the ramp, one rect per color
    rect(0.02, y[-(n + 1)], 0.34, y[-1], col = info$cols, border = NA)
    rect(0.02, y[1], 0.34, y[n + 1], border = "black")
    at <- pretty(info$zl, 4)
    at <- at[at >= info$zl[1] & at <= info$zl[2]]
    ya <- 0.12 + 0.76 * (at - info$zl[1]) / diff(info$zl)
    segments(0.34, ya, 0.42, ya)
    ## xpd = NA: labels may spill out of this narrow cell, into the next panel's margin
    text(
        0.46,
        ya,
        formatC(at, format = "g", digits = 2),
        adj = 0,
        cex = 0.6,
        xpd = NA
    )
    par(op)
}

panel <- function(
    M,
    title,
    xlabs = NULL,
    ylabs = NULL,
    annot = FALSE,
    mar = c(2.5, 2.5, 2.5, 0.5)
) {
    op <- par(mar = mar)
    info <- show_mat(M, title, xlabs, ylabs, annot)
    par(op)
    color_bar(info)
}

lat <- paste0("x", seq_len(dx)) # one label per latent dimension
## Each matrix gets its own narrow colorbar column, so the layout alternates
## panel / colorbar. On the bottom row C keeps one panel column and R spans five.
layout(
    matrix(
        c(1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 11, 11, 11, 11, 12),
        2,
        8,
        byrow = TRUE
    ),
    widths = c(2, 0.6, 2, 0.6, 2, 0.6, 2, 0.6),
    heights = c(1, 1.7)
)
par(oma = c(0, 0, 3, 0))
panel(matrix(m_0, ncol = 1), "m_0  (prior mean)", "", lat, annot = TRUE)
panel(V_0, "V_0  (prior cov.)", lat, lat, annot = TRUE)
panel(A, "A  (dynamics)", lat, lat, annot = TRUE)
panel(Q, "Q  (process noise)", lat, lat, annot = TRUE)
panel(C, sprintf("C  (loadings, %d x %d)", dy, dx), lat, NULL)
panel(R, sprintf("R  (obs. noise, %d x %d)", dy, dy))
mtext(
    "Model parameters theta = (A, Q, C, R, m_0, V_0) -- each panel on its own color scale",
    outer = TRUE,
    cex = 1.0,
    line = 0.5
)
layout(1)
par(oma = c(0, 0, 0, 0))

## ------------------------------------------------------------------ Figure 1b
## Eigenvalues of A, and the latent phase portrait.
options(repr.plot.width = 12, repr.plot.height = 5)
set.seed(123)
x_demo <- matrix(0, dx, 300) # one trajectory from the true model
xs <- MASS::mvrnorm(1, m_0, V_0)
for (t in 1:300) {
    if (t > 1) {
        xs <- as.vector(A %*% xs + MASS::mvrnorm(1, rep(0, dx), Q))
    }
    x_demo[, t] <- xs
}

op <- par(mfrow = c(1, 2), mar = c(4, 4, 3, 1))
phi <- seq(0, 2 * pi, length.out = 200)
plot(
    cos(phi),
    sin(phi),
    type = "l",
    lty = 2,
    asp = 1,
    xlim = c(-1.15, 1.15),
    ylim = c(-1.15, 1.15),
    xlab = "Re",
    ylab = "Im",
    main = "Eigenvalues of A (unit circle)"
)
abline(h = 0, v = 0, col = "gray70")
points(Re(eig_true), Im(eig_true), pch = 19, cex = 1.3, col = "firebrick")

plot(
    x_demo[1, ],
    x_demo[2, ],
    type = "l",
    col = "steelblue",
    xlab = "x1",
    ylab = "x2",
    main = sprintf("Latent trajectory (%d-D, projected on x1-x2)", dx)
)
points(x_demo[1, 1], x_demo[2, 1], pch = 19, col = "forestgreen")
points(x_demo[1, 300], x_demo[2, 300], pch = 19, col = "firebrick")
legend(
    "topright",
    c("start", "end"),
    pch = 19,
    col = c("forestgreen", "firebrick"),
    bty = "n"
)
par(op)
options(old_repr) # hand the canvas size back to whatever it was

In [ ]:
## MARSS fits ONE multivariate series (rows = channels, columns = time), not a
## batch of independent sequences. So we generate one long training series and
## one long validation series from the ground-truth model.
simulate_lds <- function(A, Q, C, R, m_0, V_0, Tlen, seed) {
    set.seed(seed)
    dx <- nrow(A)
    dy <- nrow(C)
    X <- matrix(0, dx, Tlen)
    Y <- matrix(0, dy, Tlen)
    x <- MASS::mvrnorm(1, m_0, V_0) # x_1 ~ N(m_0, V_0)
    for (t in seq_len(Tlen)) {
        if (t > 1) {
            x <- as.vector(A %*% x + MASS::mvrnorm(1, rep(0, dx), Q))
        }
        X[, t] <- x
        Y[, t] <- as.vector(C %*% x + MASS::mvrnorm(1, rep(0, dy), R))
    }
    list(states = X, obs = Y)
}

T_train <- 2000L
T_val <- 1000L
train <- simulate_lds(A, Q, C, R, m_0, V_0, T_train, seed = 10)
val <- simulate_lds(A, Q, C, R, m_0, V_0, T_val, seed = 999)
Y_train <- train$obs
X_train <- train$states # dy x T ,  dx x T
Y_val <- val$obs
X_val <- val$states
cat("Y_train:", dim(Y_train), "  Y_val:", dim(Y_val), "\n")

## 3. The latent basis is genuinely unidentifiable — a demonstration

Before fitting anything, let's *see* the symmetry. Take the true parameters, build a MARSS model with them held **fixed**, apply a random invertible $T$, and check two things on held-out data: (i) the marginal log-likelihood is unchanged, and (ii) the smoothed latents of the two models are related by exactly $T$.

*MARSS trick:* passing **numeric matrices** (rather than the shortcut strings) fixes those parameters. A model whose parameters are all fixed needs no fitting — `MARSS()` just runs the Kalman filter/smoother and reports the log-likelihood (convergence code 3 = "all parameters fixed").

In [ ]:
## Build a MARSS model with FIXED (known) parameters, then read off logLik + smoother.
## Note the letter mapping:  our A -> MARSS B ,  our C -> MARSS Z ,
## MARSS's own U (state intercept) and A (obs intercept) are fixed to zero.
build_fixed <- function(A, Q, C, R, m_0, V_0, Y, tinitx = 1L) {
    model <- list(
        B = A,
        U = "zero",
        Q = Q,
        Z = C,
        A = "zero",
        R = R,
        x0 = matrix(m_0, ncol = 1),
        V0 = V_0,
        tinitx = tinitx
    )
    MARSS(Y, model = model, silent = TRUE)
}

## apply a random invertible change of latent basis  x -> T x
Tm <- matrix(rnorm(dx * dx), dx, dx)
Tinv <- solve(Tm)

fit_true <- build_fixed(A, Q, C, R, m_0, V_0, Y_val)
fit_tf <- build_fixed(
    Tm %*% A %*% Tinv,
    Tm %*% Q %*% t(Tm),
    C %*% Tinv,
    R,
    as.vector(Tm %*% m_0),
    Tm %*% V_0 %*% t(Tm),
    Y_val
)

ll_true <- as.numeric(logLik(fit_true))
ll_tf <- as.numeric(logLik(fit_tf))
cat(sprintf("marginal log-lik (val):    true model = %.4f\n", ll_true))
cat(sprintf("                      T-transformed = %.4f\n", ll_tf))
cat(sprintf(
    "                         difference = %.2e\n",
    abs(ll_true - ll_tf)
))

## (ii) smoothed latents map onto each other by exactly T
s0 <- fit_true$states # dx x T, smoothed (true basis)
s1 <- fit_tf$states # dx x T, smoothed (T-transformed basis)
cat(sprintf(
    "\nmax | smoothed(T-model) - T . smoothed(true) | = %.2e\n",
    max(abs(s1 - Tm %*% s0))
))

Identical likelihood; latents that map onto each other by $T$. The latent coordinate system is a *modeling choice*, not something the data determine. So from here on we compare **invariants**, or we **align** before comparing.

## 4. Fit a fresh model with EM

We now discard the truth and fit $\theta$ from $Y_\text{train}$ alone. Three practical notes:

- **Stability, and what we constrain.** Plain MLE-EM for an LDS is prone to a covariance losing positive-definiteness and diverging. The Python notebook tames this with a conjugate (matrix-normal–inverse-Wishart) prior; MARSS's `kem` has no prior, so we play the same role with a **mild structural constraint** — leaving the dynamics `B` and loadings `Z` **unconstrained** while restricting both noise covariances to `"diagonal and unequal"`. That matches the generating $Q$ exactly and is honestly **misspecified** for $R$, whose truth is spatially correlated; expect Section 8 to score $R$ noticeably worse than $A$, $C$, $Q$. Because a diagonal $Q$ is not preserved by a general $T$, this also reduces the basis symmetry from the full $\mathrm{GL}(d_x)$ to the orthogonal group — but it leaves **every invariant we care about** (eigenvalues, Hankel spectrum, aligned latents, model order, the balanced realization) untouched. *(To chase the full-$\mathrm{GL}$ story exactly, set `Q = "unconstrained"` — expect slower, touchier convergence.)*
- **Warm start.** We initialize the emission matrix $Z$ from the top principal components of the data — a cheap, standard warm start passed via `inits`. `fit_lds()` and the "initial model" of Sections 7–8 are handed the *same* matrices, so the pre-EM comparison really is EM's starting point.
- **Convergence.** EM increases its objective **monotonically**; with `control = list(trace = 1)` MARSS records the per-iteration log-likelihood in `fit$iter.record$logLik`, which we plot. An **unconstrained `B` is the slow case** for EM — if the trace hasn't flattened, raise `maxit` or switch to `method = "BFGS"`.

In [ ]:
## PCA warm start for the emission matrix Z (top d_x principal directions of Y)
pca_init_Z <- function(Y, dx) {
    Yc <- Y - rowMeans(Y) # center each channel
    svd(t(Yc))$v[, 1:dx, drop = FALSE] # (dy x dx)
}

## An *unconstrained Z of a chosen size*. MARSS reads the state dimension m off
## the columns of Z, so the shortcut Z = "unconstrained" would silently set
## m = dy and ignore dx. Handing it a dy x dx matrix of distinct parameter names
## makes every entry free AND pins m = dx. Names are zero-padded so that their
## sort order matches column-major (vec) order -- which is the order the init
## vector below has to be in.
free_Z <- function(dy, dx) {
    matrix(sprintf("z%04d", seq_len(dy * dx)), dy, dx)
}

## The pre-EM starting point, as explicit matrices. fit_lds() below hands EM
## exactly these, so Sections 7 and 8 can score "the initial model" against the
## truth and show how far EM actually travels.
init_params <- function(Y, dx) {
    list(
        B = 0.5 * diag(dx), # gentle, featureless dynamics
        Q = diag(dx),
        Z = pca_init_Z(Y, dx), # PCA loadings
        R = mean(apply(Y, 1, var)) * diag(nrow(Y))
    )
}

fit_lds <- function(Y, dx, maxit = 200L, trace = 1L) {
    model <- list(
        B = "unconstrained", # m x m, with m = dx taken from Z below
        U = "zero",
        Q = "diagonal and unequal",
        Z = free_Z(nrow(Y), dx),
        A = "zero",
        R = "diagonal and unequal",
        tinitx = 1L
    )
    ## MARSS wants inits in *vectorized free-parameter* form: one (n_free x 1)
    ## column vector per matrix, in the same order as par$<matrix>. For the
    ## unconstrained B and Z that is column-major (vec) order; for the two
    ## "diagonal and unequal" covariances it is the diagonal, in order.
    p0 <- init_params(Y, dx)
    inits <- list(
        B = matrix(as.vector(p0$B), ncol = 1),
        Q = matrix(diag(p0$Q), ncol = 1),
        Z = matrix(as.vector(p0$Z), ncol = 1),
        R = matrix(diag(p0$R), ncol = 1)
    )
    MARSS(
        Y,
        model = model,
        inits = inits,
        method = "kem",
        control = list(maxit = maxit, trace = trace, safe = TRUE),
        silent = TRUE
    )
}

fit <- fit_lds(Y_train, dx, maxit = 200L)

## EM increases the log-likelihood monotonically
ll_trace <- fit$iter.record$logLik
cat("objective monotone non-decreasing:", all(diff(ll_trace) >= -1e-4), "\n")
cat(sprintf(
    "train log-lik per time step:  true=%.2f (ref)   fit=%.2f\n",
    as.numeric(logLik(build_fixed(A, Q, C, R, m_0, V_0, Y_train))) / T_train,
    as.numeric(logLik(fit)) / T_train
))

plot(
    ll_trace,
    type = "l",
    col = "steelblue",
    lwd = 2,
    xlab = "EM iteration",
    ylab = "log-likelihood",
    main = "EM converges monotonically"
)
grid()

## the pre-EM initialization, kept for the "how far did EM travel" comparisons
## in Sections 7 and 8 -- built by the same function fit_lds() started from
p_init <- init_params(Y_train, dx)

## (A, C, Q, R) from a fitted MARSS object, in whatever basis it happens to use
raw_pars <- function(mfit) {
    p <- coef(mfit, type = "matrix")
    list(A = p$B, C = p$Z, Q = p$Q, R = p$R)
}

## 5. Recovering the latent trajectory — by alignment

The fitted latents live in the model's own arbitrary basis. To compare them to the ground truth we find the linear map $T_\text{map}$ that best sends fitted smoothed means onto the true latents — precisely *undoing* the $\mathrm{GL}$ ambiguity, estimated by least squares. Any mismatch that survives alignment is **estimation/filtering error** (the latent is only partially determined by noisy data), *not* the basis symmetry.

In [ ]:
Xhat <- fit$states # dx x T, smoothed latents (fitted basis)

## least-squares map sending fitted latents onto the true latents:
##    X_true  ~  T_map %*% X_hat     (undoes the GL basis ambiguity)
T_map <- X_train %*% t(Xhat) %*% solve(Xhat %*% t(Xhat))
X_aligned <- T_map %*% Xhat
R2 <- 1 -
    sum((X_train - X_aligned)^2) /
        sum((X_train - rowMeans(X_train))^2)
cat(sprintf("alignment R^2 (fitted -> true latents): %.3f\n", R2))

win <- 1:min(250, ncol(X_train))
op <- par(mfrow = c(dx, 1), mar = c(3, 4, 2, 1))
for (i in 1:dx) {
    plot(
        win,
        X_train[i, win],
        type = "l",
        lwd = 2,
        col = "black",
        xlab = if (i == dx) "time step" else "",
        ylab = paste("latent", i),
        main = if (i == 1) {
            sprintf(
                "Recovered latent trajectory after alignment  (R^2 = %.2f)",
                R2
            )
        } else {
            ""
        }
    )
    lines(win, X_aligned[i, win], lwd = 2, lty = 2, col = "firebrick")
    if (i == 1) {
        legend(
            "topright",
            c("true", "fitted (aligned)"),
            col = c("black", "firebrick"),
            lty = c(1, 2),
            lwd = 2,
            bty = "n",
            ncol = 2
        )
    }
}
par(op)

The same fit, viewed in observation space. This is the one comparison that needs **no gauge fixing**: the basis ambiguity cancels in $(CT^{-1})(Tx)=Cx$, so predicted observations are directly comparable to the data. Two bands are shown, and the gap between them is the point — the *signal* band $CV_{t\mid T}C^\top$ (how well the noise-free readout is known) is far tighter than the *predictive* band $CV_{t\mid T}C^\top + R$ (where a measured $y_t$ should land). Almost all the visible scatter of $y$ about the fit is observation noise, not estimation error. Coverage of the 95% predictive interval is a calibration check: it should come out near 0.95 — and where it falls short, the diagonal $R$ we asked MARSS to estimate is the prime suspect, since it cannot represent the correlated noise that actually generated the data.

*MARSS note:* `MARSSkf(fit)` returns the smoothed means `xtT` ($d_x\times T$) and covariances `VtT` ($d_x\times d_x\times T$); `fit$states` is just `xtT`. To score the *validation* series under the trained parameters we refit a model on `Y_val` with every parameter fixed — the same convergence-code-3 trick as Section 3.

In [ ]:
## Observations reconstructed from the fitted latents. Note there is NO alignment here:
## C_fit %*% x_fit lives in observation space, where the GL gauge cancels, (C T^-1)(T x) = C x.
## Two bands, because there are two questions:
##   signal band     C V_{t|T} C'        -- how well do we know the noise-free readout C x_t?
##   predictive band C V_{t|T} C' + R    -- where should the *measured* y_t actually land?
pf <- raw_pars(fit)
C_fit <- pf$C
R_fit <- pf$R

predict_obs <- function(mfit) {
    kf <- MARSSkf(mfit) # smoothed means xtT (dx x T) and covariances VtT (dx x dx x T)
    Xs <- kf$xtT
    Vs <- kf$VtT
    Yh <- C_fit %*% Xs # dy x T   (the observation intercept A is fixed to zero)
    sig_var <- sapply(
        seq_len(ncol(Xs)),
        function(t) rowSums((C_fit %*% Vs[,, t]) * C_fit)
    ) # diag(C V C') per time point, dy x T
    list(Yh = Yh, sig = sig_var, pred = sig_var + diag(R_fit))
}

## every parameter fixed at the trained values -> MARSS just smooths / scores Y_val
fix_all <- function(p, tinitx = 1L) {
    list(
        B = p$B,
        U = p$U,
        Q = p$Q,
        Z = p$Z,
        A = p$A,
        R = p$R,
        x0 = p$x0,
        V0 = p$V0,
        tinitx = tinitx
    )
}
cf <- coef(fit, type = "matrix")
fit_on_val <- MARSS(Y_val, model = fix_all(cf), silent = TRUE)

po_tr <- predict_obs(fit)
po_va <- predict_obs(fit_on_val)

obs_scores <- function(Y, Yh, pred_var) {
    r2 <- 1 - sum((Y - Yh)^2) / sum((Y - rowMeans(Y))^2)
    cover <- mean(abs(Y - Yh) <= 2 * sqrt(pred_var)) # nominal 0.954
    c(r2 = r2, cover = cover)
}

for (nm in c("train", "val  ")) {
    z <- if (nm == "train") {
        obs_scores(Y_train, po_tr$Yh, po_tr$pred)
    } else {
        obs_scores(Y_val, po_va$Yh, po_va$pred)
    }
    cat(sprintf(
        "%s:  R^2(y, y_hat) = %.3f   95%% predictive coverage = %.3f\n",
        nm,
        z["r2"],
        z["cover"]
    ))
}

band_pred <- adjustcolor("firebrick", alpha.f = 0.15)
band_sig <- adjustcolor("firebrick", alpha.f = 0.40)

n_ch <- 5L
t_max <- 120L
tt <- seq_len(t_max)
op <- par(mfrow = c(n_ch, 1), mar = c(2.2, 4, 1.2, 1), oma = c(6, 0, 3, 0))
for (i in seq_len(n_ch)) {
    mu_y <- po_tr$Yh[i, tt]
    sd_p <- sqrt(po_tr$pred[i, tt])
    sd_s <- sqrt(po_tr$sig[i, tt])
    yl <- range(c(mu_y - 2 * sd_p, mu_y + 2 * sd_p, Y_train[i, tt]))
    plot(
        tt,
        Y_train[i, tt],
        type = "n",
        ylim = yl,
        xlab = "",
        ylab = paste("channel", i)
    )
    polygon(
        c(tt, rev(tt)),
        c(mu_y - 2 * sd_p, rev(mu_y + 2 * sd_p)),
        col = band_pred,
        border = NA
    )
    polygon(
        c(tt, rev(tt)),
        c(mu_y - 2 * sd_s, rev(mu_y + 2 * sd_s)),
        col = band_sig,
        border = NA
    )
    lines(tt, Y_train[i, tt], col = "black", lwd = 1.2)
    lines(tt, mu_y, col = "firebrick", lwd = 1.6)
}
mtext("time step", side = 1, line = 1.8)
z_tr <- obs_scores(Y_train, po_tr$Yh, po_tr$pred)
mtext(
    sprintf(
        paste0(
            "Observations reconstructed from the fitted latents -- no alignment ",
            "needed  (R^2 = %.2f, coverage = %.2f)"
        ),
        z_tr["r2"],
        z_tr["cover"]
    ),
    outer = TRUE,
    line = 1.2,
    cex = 1.0
)
par(op)
## one shared legend, overlaid across the full device in the bottom outer margin
op2 <- par(
    fig = c(0, 1, 0, 1),
    mfrow = c(1, 1),
    mar = c(0, 0, 0, 0),
    oma = c(0, 0, 0, 0),
    new = TRUE
)
plot.new()
legend(
    "bottom",
    horiz = TRUE,
    bty = "n",
    cex = 0.8,
    legend = c(
        "+-2sd predictive (C V C' + R)",
        "+-2sd signal (C V C')",
        "observed y",
        "C xhat (smoothed)"
    ),
    pch = c(15, 15, NA, NA),
    pt.cex = 1.8,
    lty = c(NA, NA, 1, 1),
    lwd = c(NA, NA, 1.2, 1.6),
    col = c(band_pred, band_sig, "black", "firebrick")
)
par(op2)

## 6. Invariant #1 — the eigenvalues of $A$

Because an equivalent model has $A' = T A T^{-1}$, and similarity transforms preserve eigenvalues, $\operatorname{eig}(A)$ is **identifiable** even though $A$ itself is not. So the entrywise distance $\lVert A-\hat A\rVert$ can be large while the eigenvalues coincide. The cell makes the second half of that claim directly: it hits $\hat A$ with a random invertible $T$ and re-computes the spectrum, which comes back unchanged to machine precision.

In [ ]:
A_fit <- raw_pars(fit)$A # fitted dynamics matrix (MARSS B)
eig_fit <- sort_complex(eigen(A_fit)$values)

cat("true eig(A):\n")
print(eig_true)
cat("fit  eig(A):\n")
print(eig_fit)
cat(sprintf(
    "\nentrywise ||A - A_fit||_F        = %.3f   (large: different matrices)\n",
    norm(A - A_fit, "F")
))
cat(
    "eigenvalue error |lam - lam_fit| =",
    round(Mod(eig_true - eig_fit), 4),
    "  (small: same spectrum)\n"
)

Tmap <- matrix(rnorm(dx * dx), dx, dx)
eig_T <- sort_complex(eigen(Tmap %*% A_fit %*% solve(Tmap))$values)
cat(
    "eigenvalue error after random transformation =",
    format(Mod(eig_T - eig_fit), digits = 3),
    "  (zero: same spectrum, different basis)\n"
)

phi <- seq(0, 2 * pi, length.out = 200)
plot(
    cos(phi),
    sin(phi),
    type = "l",
    lty = 2,
    asp = 1,
    xlim = c(-1.15, 1.15),
    ylim = c(-1.15, 1.15),
    xlab = "Re",
    ylab = "Im",
    main = "Eigenvalues of A: recovered"
)
abline(h = 0, v = 0, col = "gray70")
points(Re(eig_true), Im(eig_true), cex = 2.2, lwd = 2, col = "steelblue")
points(Re(eig_fit), Im(eig_fit), pch = 19, col = "firebrick")
legend(
    "bottomleft",
    c("true", "fitted"),
    col = c("steelblue", "firebrick"),
    pch = c(1, 19),
    pt.lwd = 2,
    bty = "n"
)

## 7. Invariant #2 — the Hankel matrix and the model order

A second, deeper invariant comes from the observations' own second-order statistics — and unlike the eigenvalues of $A$, you can build it without a model at all.

**Start from the data.** For a stationary process, the lag-$k$ output covariance
$$
\Lambda_k \;=\; \mathbb E\!\left[y_{t+k}\,y_t^\top\right] \qquad (k\ge 1)
$$
is something you can estimate from any recording in three lines. Stack the lags into a block-**Hankel** matrix whose $(i,j)$ block is $\Lambda_{i+j-1}$:
$$
\mathcal H_m=\begin{bmatrix}\Lambda_1&\Lambda_2&\cdots&\Lambda_m\\ \Lambda_2&\Lambda_3&&\vdots\\ \vdots&&\ddots&\\ \Lambda_m&\cdots&&\Lambda_{2m-1}\end{bmatrix}.
$$

**The model factors it.** Let $\Pi$ solve the discrete Lyapunov equation $\Pi = A\Pi A^\top + Q$ (the stationary latent covariance) and set $G = A\Pi C^\top$. Then $\Lambda_k = C A^{\,k-1} G$, so each block splits,
$$
\Lambda_{i+j-1} \;=\; C A^{\,i+j-2} G \;=\; \underbrace{\big(C A^{\,i-1}\big)}_{\text{depends only on }i}\;\underbrace{\big(A^{\,j-1} G\big)}_{\text{depends only on }j},
$$
and therefore so does the whole matrix — into a **tall** factor times a **wide** one:
$$
\mathcal H_m=\underbrace{\begin{bmatrix}C\\ CA\\ \vdots\\ CA^{m-1}\end{bmatrix}}_{\textbf{future matrix }\ \mathcal O_m}\;
\underbrace{\big[\,G\ \ AG\ \ \cdots\ \ A^{m-1}G\,\big]}_{\textbf{past matrix }\ \mathcal C_m}.
$$
$\mathcal O_m$ describes how the current state is broadcast into the next $m$ observations — the **future**. $\mathcal C_m$ describes how the previous $m$ observations deposit into the current state — the **past**. *(These are the* observability *and* controllability *matrices of the systems literature. There is no controller and no input anywhere in this model, so we use the plainer names.)*

Each factor has only $d_x$ columns or rows, so $\operatorname{rank}\mathcal H_m = d_x$: **the number of non-zero Hankel singular values is the state dimension.** Those singular values are basis-invariant (they are functions of $y$ alone), so the true model, the fitted model, and even the *raw sample covariances* should share the same rank-$d_x$ fingerprint. This is the seed of subspace identification (Ho–Kalman / N4SID).

**And there is the gauge, in one line.** The factorization is not unique. For any invertible $T$,
$$
\mathcal H_m \;=\; \big(\mathcal O_m T^{-1}\big)\big(T\,\mathcal C_m\big)
$$
is an equally good split into future times past. **The product is what the data pin down; the factors are what EM returns.** That is the identifiability problem of Section 1, now visible as a property of a single matrix — and Section 8 resolves it by choosing the split canonically.

Because $\mathcal H_m$ is assembled purely from output covariances it is **gauge-free** — no alignment enters anywhere — and for minimal models it is *complete*: two models produce the same $\mathcal H_m$ if and only if they are the same system up to a change of latent basis. So $\lVert\mathcal H_m - \hat{\mathcal H}_m\rVert_F / \lVert\mathcal H_m\rVert_F$ is the whole-system counterpart of the per-matrix errors in Section 8, and unlike those it makes a single joint statement about $(A, C, Q)$. We report it for the fitted model, the pre-EM initialization, and the raw sample covariances.

*(Base R has no `solve_discrete_lyapunov`, so the setup cell defines `dlyap()` by vectorization: $\operatorname{vec}\Pi = (I - A\otimes A)^{-1}\operatorname{vec}Q$.)*

In [ ]:
hankel_from_params <- function(A_, C_, Q_, R_, m = 10) {
    Pi <- dlyap(A_, Q_)
    G <- A_ %*% Pi %*% t(C_)
    d <- nrow(C_)
    Lam <- vector("list", 2 * m - 1) # Lam_1 .. Lam_{2m-1}
    Ak <- diag(nrow(A_)) # A^0
    for (k in seq_len(2 * m - 1)) {
        Lam[[k]] <- C_ %*% Ak %*% G
        Ak <- Ak %*% A_
    }
    H <- matrix(0, d * m, d * m)
    for (i in 1:m) {
        for (j in 1:m) {
            H[((i - 1) * d + 1):(i * d), ((j - 1) * d + 1):(j * d)] <- Lam[[
                i + j - 1
            ]]
        }
    }
    H
}

hankel_from_data <- function(Y, m = 10) {
    Yc <- Y - rowMeans(Y)
    Tt <- ncol(Yc)
    d <- nrow(Yc)
    lam <- function(k) {
        (Yc[, (k + 1):Tt, drop = FALSE] %*% t(Yc[, 1:(Tt - k), drop = FALSE])) /
            (Tt - k)
    }
    Lam <- lapply(1:(2 * m - 1), lam)
    H <- matrix(0, d * m, d * m)
    for (i in 1:m) {
        for (j in 1:m) {
            H[((i - 1) * d + 1):(i * d), ((j - 1) * d + 1):(j * d)] <- Lam[[
                i + j - 1
            ]]
        }
    }
    H
}

xcov_lag <- 10L
pf <- raw_pars(fit)
H_true <- hankel_from_params(A, C, Q, R, m = xcov_lag)
H_fit <- hankel_from_params(pf$A, pf$C, pf$Q, pf$R, m = xcov_lag)
H_init <- hankel_from_params(
    p_init$B,
    p_init$Z,
    p_init$Q,
    p_init$R,
    m = xcov_lag
)
H_emp <- hankel_from_data(Y_train, m = xcov_lag)
svals <- function(H) svd(H)$d
sv_true <- svals(H_true)
sv_fit <- svals(H_fit)
sv_emp <- svals(H_emp)

## The Hankel matrix is built only from output covariances, so it is gauge-free: no
## alignment is involved, and two minimal models share it iff they are the same system
## up to a change of latent basis.
cat(
    "relative Frobenius distance to the true Hankel matrix (no alignment needed):\n"
)
for (nm in c("fitted model", "initial model", "raw sample covariances")) {
    H <- switch(
        nm,
        "fitted model" = H_fit,
        "initial model" = H_init,
        "raw sample covariances" = H_emp
    )
    cat(sprintf("   %-24s%8.3f\n", nm, rel_fro(H_true, H)))
}
cat("\n")

k <- 1:xcov_lag
plot(
    k,
    sv_true[k] / sv_true[1],
    type = "b",
    pch = 19,
    log = "y",
    col = "steelblue",
    ylim = c(1e-3, 2),
    xlab = "singular value index",
    ylab = expression(sigma[i] / sigma[1]),
    main = "Hankel singular values: a cliff at the true state dimension"
)
lines(
    k,
    sv_fit[k] / sv_fit[1],
    type = "b",
    pch = 15,
    lty = 2,
    col = "firebrick"
)
lines(
    k,
    sv_emp[k] / sv_emp[1],
    type = "b",
    pch = 17,
    lty = 3,
    col = "darkgreen"
)
abline(v = dx, col = "firebrick", lty = 3)
text(dx + 0.15, 3e-1, sprintf("d_x = %d", dx), col = "firebrick", adj = 0)
legend(
    "topright",
    c("true model", "fitted model", "raw sample covariances"),
    col = c("steelblue", "firebrick", "darkgreen"),
    pch = c(19, 15, 17),
    lty = c(1, 2, 3),
    bty = "n"
)
cat("normalized sv (true model):", round(sv_true[1:6] / sv_true[1], 4), "\n")

## 8. Parameter recovery: comparing every matrix, in a canonical basis

Section 5 removed the gauge by *estimating* it, regressing fitted latents on true ones to get $\hat T$. That is fine for looking at trajectories, but any comparison built on it inherits whatever error is in $\hat T$. Here we instead put each model into a canonical basis on its own, then compare the results.

**Two Gram matrices.** Collapse the Section 7 factors onto the latent space, the tall future one from the left and the wide past one from the right:
$$
\Omega_{\rightarrow} \;=\; \mathcal O_m^\top \mathcal O_m \;=\; \sum_{k}(A^\top)^k\,C^\top C\,A^k,
\qquad
\Omega_{\leftarrow} \;=\; \mathcal C_m \mathcal C_m^\top \;=\; \sum_{k}A^k\,GG^\top\,(A^\top)^k .
$$
Both are $d_x\times d_x$. $u^\top\Omega_{\rightarrow}u$ says how loudly latent direction $u$ shows up in the **future** of the data, $u^\top\Omega_{\leftarrow}u$ how loudly it shows up in the **past**. *(These are the* observability *and* controllability *Gramians of the systems literature.)*

**Why an invariant exists.** The gauge acts on the two factors oppositely, $\mathcal O_m \mapsto \mathcal O_m T^{-1}$ and $\mathcal C_m \mapsto T\,\mathcal C_m$, so
$$
\Omega_{\rightarrow}\mapsto T^{-\top}\Omega_{\rightarrow}T^{-1},
\qquad
\Omega_{\leftarrow}\mapsto T\,\Omega_{\leftarrow}T^{\top},
\qquad
\Omega_{\leftarrow}\Omega_{\rightarrow}\;\mapsto\;T\big(\Omega_{\leftarrow}\Omega_{\rightarrow}\big)T^{-1}.
$$
The product is only conjugated, and eigenvalues survive conjugation, so $\sqrt{\operatorname{eig}(\Omega_{\leftarrow}\Omega_{\rightarrow})}$ is gauge-free: these are the Hankel singular values $\sigma_1\ge\cdots\ge\sigma_{d_x}$ of Section 7.

**Balancing.** Take the SVD $\mathcal H_m = U\Sigma V^\top$. Every valid split into future times past has the form $\mathcal O_m = U\Sigma^{1/2}T^{-1}$, $\mathcal C_m = T\,\Sigma^{1/2}V^\top$, which makes the gauge concrete: **it is the freedom in how you divide $\Sigma$ between past and future**, and EM returns an arbitrary division. The **balanced** realization takes $T=I$, giving each side $\sqrt\Sigma$, so that $\Omega_{\rightarrow}=\Omega_{\leftarrow}=\Sigma$. Two models related by a change of basis share $\mathcal H_m$, hence share $U\Sigma V^\top$, hence land on identical balanced parameters. The construction uses one model at a time, so nothing inherits error from a fitted $\hat T$. Each matrix then gets one number,
$$
\text{rel. error} \;=\; \lVert \hat P - P\rVert_F \,/\, \lVert P\rVert_F,
$$
zero exactly when the two matrices agree. $R$ never sees the latent basis and needs no transformation.

**The residual freedom.** Balancing does not quite pin down $T$. If $T$ and $\tilde T$ both balance the same model, $S=\tilde T T^{-1}$ satisfies both $S\Sigma S^\top=\Sigma$ and $S^\top\Sigma S=\Sigma$, which forces $S$ orthogonal and commuting with $\Sigma$: block diagonal, one block per *repeated* singular value. Distinct sorted $\sigma$ leave only $S=\operatorname{diag}(\pm1)$, sign flips and no permutations; a $\sigma$ of multiplicity $k$ leaves a full $O(k)$. Oscillatory modes are the awkward case, since a rotational pair produces two nearly equal $\sigma$. We therefore group $\sigma$'s within a relative tolerance and fit an orthogonal Procrustes on the balanced loadings inside each group, which for a block of size 1 is just a sign flip. With only *nearly* equal $\sigma$ the exact group is still $\operatorname{diag}(\pm1)$, so allowing $O(k)$ is a deliberate slackening to absorb the ill-conditioning of the eigenvector step, not an extra symmetry of the model. The cell prints the block sizes and the smallest relative $\sigma$ gap; conditioning scales like 1/gap.

**Two implementation notes.** We weight $\Omega_{\rightarrow}$ with $C^\top C$ rather than $C^\top R^{-1}C$. The latter is the Fisher information the observations carry about the state and orders directions by estimability rather than raw output energy, but it makes the basis depend on $\hat R$ and lets error there leak into the $A,C,Q$ comparison. And the Grams are never built from stacks (at $d_y=50$, $m=10$ each is $500\times d_x$); peeling off the $k=0$ term makes each one the fixed point of a discrete Lyapunov equation,
$$
\Omega_{\rightarrow} = A^\top\Omega_{\rightarrow}A + C^\top C,
\qquad
\Omega_{\leftarrow} = A\,\Omega_{\leftarrow}A^\top + GG^\top,
$$
one `solve_discrete_lyapunov` call each. This is the $m\to\infty$ limit, so the $\sigma_i$ printed here are the asymptotic values that Section 7's finite-lag ones converge to.

**What to watch.** Balancing spends all $d_x^2$ gauge parameters, so the per-matrix errors are not independent of each other, and the whole construction is well conditioned only when the $\sigma_i$ are well separated. Two self-checks print alongside the table: applying a random $T$ to the truth and re-balancing should return the same matrices (gauge invariance), and balancing an already-balanced model should change nothing (idempotence). Both should land at machine precision; if they do not, the residual freedom was resolved wrongly and every number in the table is suspect. We repeat the comparison for the pre-EM initialization to show how far EM travels. *(Heat maps share one color scale per row, so a scale error is visible rather than normalized away.)*

In [ ]:
canonical <- function(A_, C_, Q_, R_) {
    ## BALANCED realization -- the canonical gauge. Section 7 split the Hankel matrix into
    ## a tall future factor O_m and a wide past factor C_m; collapse each onto the latent
    ## space (the tall one from the left, the wide one from the right) to get two dx x dx
    ## Gram matrices:
    ##   Gram_future = O_m' O_m = sum_k (A')^k C'C A^k     how loudly a latent direction
    ##   Gram_past   = C_m C_m' = sum_k A^k GG' (A')^k     shows up in the future / the past
    ## Balancing splits the Hankel SVD down the middle, handing each side sqrt(Sigma), which
    ## makes both Grams equal to Sigma = diag(Hankel singular values). It is computed from
    ## one model alone -- no reference, no estimated alignment -- so nothing here inherits
    ## error from a fitted T. The dlyap() solves below are the m -> inf limit of those sums.
    ## Weighting note: C'C measures raw output energy. C' R^-1 C is equally valid and is the
    ## Fisher information about the state, which orders directions by estimability rather
    ## than by energy -- but it makes the basis depend on Rhat, so Rhat's error would leak
    ## into the A / C / Q comparison. We keep C'C so the basis depends only on (A, C, Q).
    d <- nrow(A_)
    Pi <- dlyap(A_, Q_) # stationary latent covariance
    G <- A_ %*% Pi %*% t(C_) # lag-1 cross-covariance
    Gram_past <- dlyap(A_, G %*% t(G))
    Gram_future <- dlyap(t(A_), t(C_) %*% C_)
    L <- t(chol((Gram_past + t(Gram_past)) / 2)) # lower triangular, L L' = Gram_past
    e <- eigen(
        t(L) %*% ((Gram_future + t(Gram_future)) / 2) %*% L,
        symmetric = TRUE
    )
    hsv <- sqrt(pmax(e$values, 1e-12)) # Hankel singular values; eigen() sorts descending
    U <- e$vectors
    Tinv <- L %*% U %*% diag(hsv^-0.5, nrow = d)
    Tb <- diag(hsv^0.5, nrow = d) %*% t(U) %*% forwardsolve(L, diag(d))
    stopifnot(max(abs(Tb %*% Tinv - diag(d))) < 1e-6) # balancing transform consistent
    list(
        P = list(
            A = Tb %*% A_ %*% Tinv,
            C = C_ %*% Tinv,
            Q = Tb %*% Q_ %*% t(Tb),
            R = R_
        ),
        hsv = hsv
    )
}

resolve <- function(P_ref, P, hsv, tol = 0.05) {
    ## Resolve the residual freedom of the balanced form against a reference.
    ## If T and T~ both balance the same model, S = T~ T^-1 obeys S Sig S' = Sig AND
    ## S' Sig S = Sig, which forces S orthogonal and commuting with Sig: block diagonal,
    ## one block per REPEATED singular value. With the sigma sorted and distinct the group
    ## is therefore just diag(+-1) -- sign flips, no permutations. We treat sigma within a
    ## relative distance `tol` as one block and allow a full O(k) rotation inside it. With
    ## only *nearly* equal sigma the exact group is still diag(+-1), so this is a deliberate
    ## slackening that absorbs the ill-conditioning of the eigenvector step, not an extra
    ## symmetry of the model. Within each block we take the orthogonal Procrustes fit of the
    ## balanced loadings; for a 1x1 block that reduces exactly to a sign flip.
    d <- ncol(P$A)
    brk <- integer(0)
    if (d > 1) {
        for (i in 2:d) {
            if ((hsv[i - 1] - hsv[i]) / hsv[i - 1] > tol) brk <- c(brk, i)
        }
    }
    cuts <- c(1L, brk, d + 1L) # block start indices, plus a sentinel
    S <- matrix(0, d, d)
    sizes <- integer(0)
    for (k in seq_len(length(cuts) - 1L)) {
        b <- cuts[k]:(cuts[k + 1L] - 1L)
        sv <- svd(t(P$C[, b, drop = FALSE]) %*% P_ref$C[, b, drop = FALSE])
        S[b, b] <- sv$u %*% t(sv$v)
        sizes <- c(sizes, length(b))
    }
    list(
        P = list(
            A = t(S) %*% P$A %*% S,
            C = P$C %*% S,
            Q = t(S) %*% P$Q %*% S,
            R = P$R
        ),
        sizes = sizes
    )
}

cn_true <- canonical(A, C, Q, R)
cn_fit <- canonical(pf$A, pf$C, pf$Q, pf$R)
cn_init <- canonical(p_init$B, p_init$Z, p_init$Q, p_init$R)
P_true <- cn_true$P
hsv_true <- cn_true$hsv
rs_fit <- resolve(P_true, cn_fit$P, hsv_true)
rs_init <- resolve(P_true, cn_init$P, hsv_true)
P_fit <- rs_fit$P
P_init <- rs_init$P

nms <- c("A", "C", "Q", "R")
labs <- c(
    A = "dynamics A",
    C = "loadings C",
    Q = "process cov Q",
    R = "obs. noise cov R"
)

cat("relative Frobenius error in the balanced (canonical) basis\n")
cat(sprintf("%-12s%10s%10s\n", "parameter", "fitted", "initial"))
for (nm in nms) {
    cat(sprintf(
        "%-12s%10.3f%10.3f\n",
        nm,
        rel_fro(P_true[[nm]], P_fit[[nm]]),
        rel_fro(P_true[[nm]], P_init[[nm]])
    ))
}

## ---- does the canonical form do what it claims? ----
gap <- (hsv_true[-dx] - hsv_true[-1]) / hsv_true[-dx]
cat("\nHankel singular values (m -> inf), true :", round(hsv_true, 3), "\n")
cat("                                 fitted :", round(cn_fit$hsv, 3), "\n")
cat(sprintf(
    "smallest relative gap sigma_i -> sigma_i+1 = %.3f   (eigenvector conditioning scales like 1/gap)\n",
    min(gap)
))
cat(
    "residual-freedom blocks resolved:",
    rs_fit$sizes,
    " (a block of size k > 1 means an O(k) rotation was fitted, not just a sign)\n"
)

set.seed(0)
Tc <- matrix(rnorm(dx * dx), dx, dx)
Tci <- solve(Tc)
cn_g <- canonical(Tc %*% A %*% Tci, C %*% Tci, Tc %*% Q %*% t(Tc), R) # same model, new gauge
P_g <- resolve(P_true, cn_g$P, hsv_true)$P
cn_i2 <- canonical(P_true$A, P_true$C, P_true$Q, P_true$R) # balance the balanced form
P_i2 <- resolve(P_true, cn_i2$P, hsv_true)$P # a fixed point only modulo the residual group
cat(sprintf(
    "gauge invariance : random T applied to the truth, re-balanced -> max |diff| = %.2e   |sigma diff| = %.2e\n",
    max(sapply(nms, function(n) max(abs(P_g[[n]] - P_true[[n]])))),
    max(abs(cn_g$hsv - hsv_true))
))
cat(sprintf(
    "idempotence      : balancing the balanced form (mod residual group) -> max |diff| = %.2e   |sigma diff| = %.2e\n",
    max(sapply(nms, function(n) max(abs(P_i2[[n]] - P_true[[n]])))),
    max(abs(cn_i2$hsv - hsv_true))
))

## ---- triplet heat maps:  generating | fitted | initial ----
## One color scale per row (set by the generating matrix), so a scale error is visible
## rather than normalized away -- the balanced basis makes the panels directly comparable.
cols <- hcl.colors(64, "Blue-Red 3")
op <- par(
    mfrow = c(length(nms), 3),
    mar = c(1, 3.5, 3, 1),
    oma = c(0, 0, 2.5, 0)
)
for (nm in nms) {
    v <- max(abs(P_true[[nm]]))
    trip <- list(P_true[[nm]], P_fit[[nm]], P_init[[nm]])
    ttl <- c(
        "generating",
        sprintf("fitted  rel.err = %.3f", rel_fro(P_true[[nm]], P_fit[[nm]])),
        sprintf("initial  rel.err = %.3f", rel_fro(P_true[[nm]], P_init[[nm]]))
    )
    for (j in 1:3) {
        P <- trip[[j]]
        image(
            x = seq_len(ncol(P)),
            y = seq_len(nrow(P)),
            z = t(P)[, nrow(P):1, drop = FALSE],
            zlim = c(-v, v),
            col = cols,
            axes = FALSE,
            xlab = "",
            ylab = "",
            main = ttl[j],
            cex.main = 0.95
        )
        box()
        if (j == 1) mtext(labs[nm], side = 2, line = 1)
    }
}
mtext(
    "Parameter recovery in the balanced basis (shared scale per row)",
    outer = TRUE,
    cex = 1.05
)
par(op)

## 9. Model selection by cross-validated log-likelihood

In practice $d_x$ is unknown. We fit models of increasing latent dimension and score each by its **held-out predictive log-likelihood** — the log-likelihood of the *validation* series under the fitted parameters, per time step. We prefer this to AIC/BIC: it measures generalization directly and needs no parameter-counting penalty (awkward here anyway, given the flat likelihood directions). The curve should rise until $d_x^\star$ and then flatten — extra latent dimensions should not improve prediction of data generated by a $d_x$-dimensional process.

*Held-out scoring in MARSS:* refit a model on `Y_val` whose parameters are **all fixed** at the trained values (convergence code 3) — MARSS then just returns that series' log-likelihood.

> **This is the slow cell.** It runs ten EM fits with an unconstrained `B`. `maxit = 100` keeps it tractable; raise it if the selected $d_x$ looks unstable.

In [ ]:
dims <- 1:10
val_ll <- numeric(length(dims))
for (i in seq_along(dims)) {
    k <- dims[i]
    fit_k <- fit_lds(Y_train, k, maxit = 100L, trace = 0L)
    p <- coef(fit_k, type = "matrix") # trained matrices
    vfit <- MARSS(Y_val, model = fix_all(p), silent = TRUE) # code 3: just scores it
    val_ll[i] <- as.numeric(logLik(vfit)) / T_val # per time step
    cat(sprintf("  d_x = %d:  val log-lik/step = %.4f\n", k, val_ll[i]))
}
best <- dims[which.max(val_ll)]

plot(
    dims,
    val_ll,
    type = "b",
    pch = 19,
    col = "steelblue",
    xlab = expression(paste("latent dimension ", d[x])),
    ylab = "validation log-lik / step",
    main = sprintf("Cross-validated model selection picks d_x = %d", best)
)
abline(v = best, col = "firebrick", lty = 3)
abline(v = dx, col = "black", lty = 2)
legend(
    "bottomright",
    c("selected", "true d_x"),
    col = c("firebrick", "black"),
    lty = c(3, 2),
    bty = "n"
)

## Recap & what's next

- A linear–Gaussian SSM is identifiable **only up to an invertible change of latent basis** ($x_t\mapsto Tx_t$). The likelihood is flat along that orbit; raw parameters and latent coordinates are not estimable.
- **What is estimable:** invariants of the orbit — the **eigenvalues of $A$**, the **Hankel singular values** (which also reveal the **model order** $d_x$) — the latent trajectory *after alignment*, and the observations themselves, which need no gauge fixing at all.
- **The full parameter set is comparable too**, but only after each model is independently put into the **balanced canonical realization**: no estimated $\hat T$, no error inherited from an alignment, and a single relative-Frobenius number per matrix that is zero iff the matrices agree.
- MARSS's EM (`method = "kem"`) recovers the model up to that symmetry; **cross-validated log-likelihood** selects $d_x$, agreeing with the Hankel cliff.

**MARSS notes for debugging.**
- `MARSS()` data is a matrix with **rows = channels, columns = time**. Our `A`→`B`, `C`→`Z`; MARSS's `A`/`U` are intercepts, fixed to zero here.
- Fitted matrices: `coef(fit, type = "matrix")$B` / `$Z` / `$Q` / `$R`. Smoothed latents: `fit$states`; means *and* covariances: `MARSSkf(fit)$xtT` and `$VtT`.
- A fully-fixed model returns its log-likelihood with no fitting (convergence code 3) — used for the held-out scores and for smoothing the validation series.
- **Model dimension comes from `Z`.** The shortcut `Z = "unconstrained"` sets $m=n$ (i.e. $d_x=d_y$); to fit a $d_x$-state model pass a `dy x dx` matrix of distinct parameter names instead.
- **`inits` are vectorized.** Each element must be an `(n_free x 1)` column vector matching `par$<matrix>` — column-major for the unconstrained `B` and `Z`, the diagonal in order for the `"diagonal and unequal"` covariances — not the matrix itself.
- Unconstrained `B` is EM's slow case: raise `control$maxit`, or try `method = "BFGS"` (optionally warm-started from a `kem` fit).
- Base R has no `solve_discrete_lyapunov`, matrix power, or spectral-norm-free balancing routine; the setup cell defines `dlyap()`, `mpow()` and `rel_fro()` once and everything else is built from `chol`, `eigen`, `svd` and `forwardsolve`.

**Next.** So far the dynamics $A$ were fixed for all time. **Notebook 3** lets the system *switch* between several linear regimes — a **switching LDS** — and fits it to resting-state fMRI, asking whether the brain's spontaneous activity is better described by one linear process or several, and what dynamical "modes" those regimes correspond to.